# Analisis Probabilitas Hujan di Nusa Tenggara Timur (NTT)
### Periode 1985 – 2015 | Data ERA5 Reanalysis × ONI NOAA

---

## Latar Belakang

Nusa Tenggara Timur (NTT) merupakan salah satu wilayah di Indonesia yang memiliki pola hujan yang sangat dipengaruhi oleh kondisi iklim global, khususnya fenomena **El Niño – La Niña** yang diukur melalui indeks **ONI (*Oceanic Niño Index*)**.

- **El Niño** (ONI tinggi, > +0.5) cenderung menyebabkan **musim kering yang lebih panjang** di NTT.
- **La Niña** (ONI rendah, < -0.5) cenderung menyebabkan **curah hujan yang lebih tinggi** dari normal.

---

## Tujuan Analisis

Notebook ini bertujuan untuk:

1. **Memproses data curah hujan** dari ERA5 Reanalysis dan data indeks ONI dari NOAA
2. **Menghitung probabilitas hujan** per musim dan per titik grid spasial di wilayah NTT
3. **Menganalisis korelasi** antara probabilitas hujan dan indeks ONI per musim
4. **Memvisualisasikan sebaran spasial** probabilitas hujan seluruh wilayah NTT
5. **Mengekspor data** untuk digunakan di dashboard interaktif

---

## Pembagian Musim

Analisis ini menggunakan 4 kelompok musim berdasarkan bulan:

| Kode | Bulan | Keterangan |
|------|-------|------------|
| **JFM** | Januari – Maret | Puncak musim hujan |
| **AMJ** | April – Juni | Transisi hujan ke kering |
| **JAS** | Juli – September | Musim kering |
| **OND** | Oktober – Desember | Awal musim hujan |

---

## Definisi Probabilitas Hujan

Probabilitas hujan dihitung dengan cara:
- Setiap hari dikategorikan: **hujan** jika curah hujan > 0.5 mm, **tidak hujan** jika ≤ 0.5 mm
- Probabilitas per musim = persentase hari hujan dari total hari dalam musim tersebut

**Contoh:** Jika dalam musim JFM (90 hari) ada 54 hari hujan → probabilitas = 60%

---
## Bagian 1: Persiapan — Import Library

**Tujuan:** Memuat semua pustaka (library) Python yang diperlukan untuk analisis data, visualisasi, dan pengolahan data spasial.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---
## Bagian 2: Load dan Pembersihan Data ONI

**Tujuan:** Membaca data indeks ONI dari NOAA, membuang kolom musim yang tidak digunakan, dan memfilter hanya periode 1985–2015.

**Kolom yang dipertahankan:** `Year`, `JFM`, `AMJ`, `JAS`, `OND`  
**Kolom yang dihapus:** `DJF`, `FMA`, `MAM`, `MJJ`, `JJA`, `ASO`, `SON`, `NDJ` (musim tumpang tindih yang tidak dipakai)

In [ ]:
data = pd.read_csv('C:/Metklim pbl/ONI.csv')
data

In [ ]:
data = data.drop(columns=['DJF', 'FMA', 'MAM', 'MJJ','ASO', 'JJA', 'SON', 'NDJ'])
data

In [ ]:
data = data[
    (data['Year'] >= 1985) &
    (data['Year'] <= 2015)
]
data

---
## Bagian 3: Konversi Data ERA5 dari Format NetCDF ke CSV

**Tujuan:** Mengubah data curah hujan ERA5 (format `.nc` / NetCDF) menjadi format tabel CSV agar bisa diproses lebih lanjut dengan pandas.

> **Catatan:** Cell ini hanya perlu dijalankan **sekali**. File `rainfallNTT.csv` akan tersimpan dan bisa langsung dipakai di cell berikutnya tanpa perlu dikonversi ulang.

In [ ]:
import xarray as xr
DS = xr.open_dataset("data_stream-oper_stepType-accum.nc")
DS.to_dataframe().to_csv("rainfallNTT.csv")

---
## Bagian 4: Load Data Curah Hujan ERA5

**Tujuan:** Membaca data curah hujan hasil konversi, lalu menyimpan data ONI ke variabel terpisah agar tidak tercampur.

In [ ]:
dataONI = data

In [ ]:
dataCH = pd.read_csv('C:/Metklim pbl/rainfallNTT.csv')
dataCH

**Tujuan:** Menghapus kolom `number` dan `expver` yang merupakan metadata teknis ERA5 dan tidak dibutuhkan untuk analisis.

In [ ]:
dataCH = dataCH.drop(columns=['number','expver'])
dataCH

---
## Bagian 5: Eksplorasi Awal Data

**Tujuan:** Memahami struktur kedua dataset — tipe data tiap kolom, jumlah baris, dan ada tidaknya nilai yang hilang (*missing values*).

In [ ]:
dataCH.info()
dataONI.info()

**Tujuan:** Mengkonversi satuan curah hujan dari meter (satuan asli ERA5) ke **milimeter (mm)** yang lebih umum digunakan dalam meteorologi.

> **Rumus:** 1 m = 1000 mm, jadi kolom `tp` dikalikan 1000.

In [ ]:
dataCH['tp'] = dataCH['tp'] * 1000

**Tujuan:** Menampilkan statistik deskriptif (nilai minimum, maksimum, rata-rata, standar deviasi, dsb.) untuk masing-masing dataset sebagai gambaran umum sebaran data.

In [ ]:
pd.set_option('display.float_format', '{:.6f}'.format)

dataCH.describe()

In [ ]:
dataONI.describe()

---
## Bagian 6: Visualisasi Time Series Indeks ONI

**Tujuan:** Melihat pola indeks ONI dari tahun 1985 hingga 2015 untuk setiap musim. Garis putus-putus di angka 0 memisahkan kondisi El Niño (di atas 0) dan La Niña (di bawah 0).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.plot(dataONI['Year'], dataONI['JFM'], label='JFM')
plt.plot(dataONI['Year'], dataONI['AMJ'], label='AMJ')
plt.plot(dataONI['Year'], dataONI['JAS'], label='JAS')
plt.plot(dataONI['Year'], dataONI['OND'], label='OND')

plt.axhline(0, linestyle='--')

plt.xlabel('Year')
plt.ylabel('ONI Index')
plt.title('Oceanic Niño Index (ONI)')
plt.legend()

plt.show()

---
## Bagian 7: Pembuatan Kategori Hujan

**Tujuan:** Mengubah nilai curah hujan (mm) menjadi kategori biner:
- `1` = **Hujan** → curah hujan > 0.5 mm
- `0` = **Tidak Hujan** → curah hujan ≤ 0.5 mm

Ambang batas 0.5 mm digunakan karena nilai di bawahnya dianggap sebagai embun/noise dan bukan hujan yang signifikan secara meteorologis.

In [ ]:
# 1 = hujan dan 0 = tidak hujan
dataCH['Hujan'] = (dataCH['tp'] > 0.5).astype(int)
dataCH

---
## Bagian 8: Perhitungan Probabilitas Hujan per Musim (Agregat Temporal)

**Tujuan:** Menghitung probabilitas hujan rata-rata seluruh NTT per tahun per musim, yang nantinya akan dikorelasikan dengan indeks ONI.

**Langkah-langkah:**
1. Parse kolom waktu `valid_time` menjadi format tanggal
2. Ekstrak tahun dan bulan
3. Kelompokkan bulan menjadi 4 musim (JFM, AMJ, JAS, OND)
4. Hitung rata-rata probabilitas hujan per tahun per musim
5. Ubah format tabel agar tiap musim menjadi kolom tersendiri

In [ ]:
import pandas as pd

# datetime
dataCH['valid_time'] = pd.to_datetime(dataCH['valid_time'])

# ubah tp ke mm kalau belum
dataCH['tp'] = dataCH['tp'] * 1000

# kategori hujan
dataCH['Hujan'] = (dataCH['tp'] > 0.5).astype(int)

# ambil tahun dan bulan
dataCH['Year'] = dataCH['valid_time'].dt.year
dataCH['Month'] = dataCH['valid_time'].dt.month

# fungsi musim
def musim(bulan):
    if bulan in [1,2,3]:
        return 'JFM'
    elif bulan in [4,5,6]:
        return 'AMJ'
    elif bulan in [7,8,9]:
        return 'JAS'
    else:
        return 'OND'

# kolom musim
dataCH['Season'] = dataCH['Month'].apply(musim)

# hitung persentase hujan
hasilCH = (
    dataCH.groupby(['Year', 'Season'])['Hujan']
    .mean()
    .mul(100)
    .reset_index()
)

# pivot
hasilCH = hasilCH.pivot(
    index='Year',
    columns='Season',
    values='Hujan'
).reset_index()

# rename kolom
hasilCH = hasilCH.rename(columns={
    'JFM': 'CH_JFM',
    'AMJ': 'CH_AMJ',
    'JAS': 'CH_JAS',
    'OND': 'CH_OND'
})

print(hasilCH)

---
## Bagian 9: Penggabungan Data CH dan ONI

**Tujuan:** Menggabungkan tabel probabilitas hujan (`hasilCH`) dengan tabel indeks ONI (`dataONI`) berdasarkan kolom `Year`, sehingga setiap baris merepresentasikan satu tahun dengan nilai CH dan ONI untuk semua musim sekaligus.

In [ ]:
gabung = pd.merge(
    hasilCH,
    dataONI,
    on='Year'
)
gabung

---
## Bagian 10: Analisis Korelasi CH vs ONI — Heatmap

**Tujuan:** Menghitung dan memvisualisasikan kekuatan hubungan antara probabilitas hujan dan indeks ONI untuk setiap musim menggunakan **korelasi Pearson**.

**Cara membaca heatmap:**
- Nilai mendekati **-1** (warna biru gelap) → hubungan **negatif kuat**: saat ONI tinggi (El Niño), hujan berkurang
- Nilai mendekati **+1** (warna merah gelap) → hubungan **positif kuat**: saat ONI tinggi, hujan bertambah
- Nilai mendekati **0** → **tidak ada hubungan** yang jelas

In [ ]:
corr_focus = gabung[
    ['CH_JFM', 'CH_AMJ', 'CH_JAS', 'CH_OND',
     'JFM', 'AMJ', 'JAS', 'OND']
].corr()

plt.figure(figsize=(8,6))

sns.heatmap(
    corr_focus,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Korelasi ONI dan Probabilitas Hujan')
plt.show()

---
## Bagian 11: Load Shapefile Batas Wilayah Indonesia

**Tujuan:** Membaca data batas wilayah administrasi Indonesia (format shapefile) yang akan digunakan sebagai overlay pada peta spasial, agar batas kabupaten/provinsi terlihat di atas peta probabilitas hujan.

In [ ]:
import geopandas as gpd

indo = gpd.read_file('indonesia_kab.shp')

---
## Bagian 12: Persiapan Data Spasial per Titik Grid

**Tujuan:** Mempersiapkan ulang kolom waktu dan kategori hujan, lalu membuat kolom musim yang akan digunakan untuk agregasi spasial.

> **Catatan:** Langkah ini melengkapi Bagian 8. Perbedaannya adalah Bagian 8 merata-rata seluruh NTT menjadi 1 nilai per tahun per musim (untuk korelasi dengan ONI), sedangkan bagian ini mempertahankan informasi per titik grid (latitude-longitude) untuk keperluan peta.

In [ ]:
import pandas as pd

# datetime
dataCH['valid_time'] = pd.to_datetime(dataCH['valid_time'])

# ubah mm
dataCH['tp'] = dataCH['tp'] * 1000

# kategori hujan
dataCH['Hujan'] = (dataCH['tp'] > 0.5).astype(int)

# tahun dan bulan
dataCH['Year'] = dataCH['valid_time'].dt.year
dataCH['Month'] = dataCH['valid_time'].dt.month

**Tujuan:** Mendefinisikan ulang fungsi pengelompokan bulan ke musim dan menerapkannya ke seluruh data.

In [ ]:
def musim(bulan):

    if bulan in [1,2,3]:
        return 'JFM'

    elif bulan in [4,5,6]:
        return 'AMJ'

    elif bulan in [7,8,9]:
        return 'JAS'

    else:
        return 'OND'

dataCH['Season'] = dataCH['Month'].apply(musim)

**Tujuan:** Menghitung probabilitas hujan per kombinasi **Tahun × Musim × Titik Grid (lat-lon)**. Hasilnya adalah tabel `spasial` yang menyimpan persentase hari hujan di setiap titik grid untuk setiap tahun dan musim — inilah data utama untuk membuat peta.

In [ ]:
spasial = (
    dataCH.groupby(
        ['Year','Season','latitude','longitude']
    )['Hujan']
    .mean()
    .mul(100)
    .reset_index()
)

---
## Bagian 13: Persiapan Peta — Import Library dan Kombinasi Tahun-Musim

**Tujuan:** Mengimport library untuk visualisasi peta dan membuat daftar semua kombinasi tahun-musim yang tersedia di data, yang akan digunakan sebagai urutan frame (halaman) peta.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
kombinasi = (
    spasial[['Year','Season']]
    .drop_duplicates()
    .values
    .tolist()
)

---
## Bagian 14: Visualisasi Peta Spasial Probabilitas Hujan

**Tujuan:** Membuat peta sebaran spasial probabilitas hujan di seluruh wilayah NTT untuk setiap kombinasi tahun dan musim.

**Cara membaca peta:**
- Warna **biru** → probabilitas hujan **tinggi** (sering hujan)
- Warna **merah/kuning** → probabilitas hujan **rendah** (jarang hujan)
- Setiap halaman gambar menampilkan **20 peta sekaligus** (5 kolom × 4 baris)
- Hasil disimpan otomatis sebagai file gambar PNG (`Frame_1.png`, `Frame_2.png`, dst.)

> **Informasi:** Total kombinasi = 31 tahun × 4 musim = **124 peta**, dibagi menjadi beberapa frame masing-masing 20 peta.

In [ ]:
from shapely.geometry import Point
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

# =====================================
# LOOP FRAME
# =====================================

for frame in range(0, len(kombinasi), 20):

    subset_kombinasi = kombinasi[frame:frame+20]

    fig, axes = plt.subplots(
        4, 5,
        figsize=(20,16)
    )

    axes = axes.flatten()

    # =====================================
    # LOOP SUBPLOT
    # =====================================

    for i, (tahun, musim) in enumerate(subset_kombinasi):

        ax = axes[i]

        # =====================================
        # SUBSET DATA
        # =====================================

        subset = spasial[
            (spasial['Year'] == tahun) &
            (spasial['Season'] == musim)
        ]

        # =====================================
        # PIVOT GRID
        # =====================================

        pivot = subset.pivot(
            index='latitude',
            columns='longitude',
            values='Hujan'
        )

        # urut latitude biar ga kebalik
        pivot = pivot.sort_index()

        # =====================================
        # GRID
        # =====================================

        lon = pivot.columns.values
        lat = pivot.index.values

        Lon, Lat = np.meshgrid(
            lon,
            lat
        )

        Z = pivot.values

        # =====================================
        # CONTOURF SPASIAL
        # =====================================

        im = ax.contourf(
            Lon,
            Lat,
            Z,
            levels=15,
            cmap='RdYlBu_r',
            extend='both'
        )

        # =====================================
        # PLOT SHAPEFILE
        # =====================================

        indo.plot(
            ax=ax,
            color='none',
            edgecolor='black',
            linewidth=0.5
        )

        # =====================================
        # BATAS WILAYAH NTT
        # =====================================

        ax.set_xlim(118,126)
        ax.set_ylim(-12,-7)

        # =====================================
        # JUDUL
        # =====================================

        ax.set_title(
            f'{tahun} - {musim}',
            fontsize=10
        )

        # =====================================
        # LABEL
        # =====================================

        ax.set_xlabel('')
        ax.set_ylabel('')

    # =====================================
    # HAPUS SUBPLOT KOSONG
    # =====================================

    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    # =====================================
    # COLORBAR
    # =====================================

    cbar = fig.colorbar(
        im,
        ax=axes,
        shrink=0.7
    )

    cbar.set_label(
        'Probabilitas Hujan (%)'
    )

    # =====================================
    # LAYOUT
    # =====================================

    plt.tight_layout()

    # =====================================
    # SAVE
    # =====================================

    plt.savefig(
        f'Frame_{frame//20 + 1}.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

---
## Bagian 15: Ekspor Data Spasial untuk Dashboard

**Tujuan:** Menyimpan tabel `spasial` (probabilitas hujan per titik grid per tahun per musim) ke file CSV, yang akan digunakan sebagai sumber data utama untuk dashboard interaktif Streamlit.

File `spasial_data.csv` ini berisi semua informasi yang dibutuhkan dashboard:
- Koordinat titik grid (`latitude`, `longitude`)
- Tahun dan musim
- Nilai probabilitas hujan (%)

In [ ]:

# Ekspor ke folder dataset untuk dashboard
spasial.to_csv('spasial_data.csv', index=False)
print("Data berhasil diekspor untuk Dashboard!")